<a href="https://colab.research.google.com/github/paddy960609/strava_stat_prediction/blob/main/strava_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Import Libraries
from google.colab import drive, auth
drive.mount('/content/gdrive')
auth.authenticate_user()
import os
os.chdir('/content/gdrive/MyDrive/strava_LastSixMonths')
import pandas as pd

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
# 2. Load Strava Data
df_activities_count = pd.read_csv('count_activities.csv')
# 3. Data Cleaning
# 4. Exploratory Data Analysis
# 5. Feature Engineering
print(df_activities_count.head())
df_activities_count_update = df_activities_count.drop(columns=['Activity', 'Month '])
print(df_activities_count_update.head())
df_activity_calories = pd.read_csv('activity_calories.csv')

print(df_activity_calories.head())
df_activity_calories.shape
df_activity_calories_update = df_activity_calories.rename(columns={"Value": "Activity_Calories"})
df_activity_calories_update = df_activity_calories_update.drop(columns=['Activity Type', 'Month '])
print(df_activity_calories_update.head())

In [ ]:

df_mean_heartrate = pd.read_csv('average_heart_rate.csv')

print(df_mean_heartrate.head())
df_mean_heartrate.shape
df_mean_heartrate_update = df_mean_heartrate.rename(columns={"Value": "Mean_Heart_Rate(bpm)"})
# columns_to_drop = ['Heart Rate', 'Month']
df_mean_heartrate_update  = df_mean_heartrate_update.drop(columns=['Heart Rate', 'Month '])
print(df_mean_heartrate_update.head())

In [ ]:
df_average_run_cadence= pd.read_csv('average_run_cadence.csv')

print(df_average_run_cadence.head())
df_average_run_cadence_update = df_average_run_cadence.rename(columns={"Value": "Average_Run_Cadence(spm)"})
df_average_run_cadence_update = df_average_run_cadence_update.drop(columns=['Run Cadence', 'Month '])
print(df_average_run_cadence_update.head())

In [ ]:
df_average_pace= pd.read_csv('average_pace.csv')

print(df_average_pace.head())

df_average_pace_update = df_average_pace.rename(columns={"Value": "Average_Pace(min/mile)"})
df_average_pace_update = df_average_pace_update.drop(columns=['Pace', 'Month '])
print(df_average_pace_update.head())


In [ ]:
df_mean_stride_length = pd.read_csv('average_stride_length.csv')

print(df_mean_stride_length.head())
df_mean_stride_length.shape
df_mean_stride_length_update = df_mean_stride_length.rename(columns={"Value": "Mean_Stride_Length(m)"})
df_mean_stride_length_update = df_mean_stride_length_update.drop(columns=['m', 'Month '])
print(df_mean_stride_length_update.head())


In [ ]:
df_VO2_max = pd.read_csv('vo2_max.csv', on_bad_lines='skip')


print(df_VO2_max.head())
df_VO2_max.shape

df_VO2_max = df_VO2_max.drop(columns=['activity ', 'Month '])
print(df_VO2_max.head())

In [ ]:
combined_df = pd.concat([df_activities_count_update, df_activity_calories_update, df_average_run_cadence_update, df_mean_stride_length_update, df_VO2_max], axis=1)
print(combined_df)

In [ ]:
# 6. Model Training

import numpy as np
from sklearn.model_selection import train_test_split
target_df = combined_df['VO₂ Max']
input_df = combined_df.drop(columns = ['VO₂ Max'])
X = np.array(input_df).astype('float32')
y = np.array(target_df).astype('float32')
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train, X_temp, y_train, y_temp = train_test_split(
    input_df.values, target_df.values, test_size=0.25, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)


In [ ]:
# Train an XGBoost classifier model
!pip install xgboost

from xgboost import XGBRegressor
model = XGBRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42)
model.fit(X_train, y_train)

In [ ]:
result_train = model.score(X_train, y_train)
result_train

In [ ]:
y_predict = model.predict(X_test)

In [ ]:
# 7. Model Evaluation
from sklearn.metrics import mean_squared_error, r2_score

rmse = mean_squared_error(y_test, y_predict)
r2 = r2_score(y_test, y_predict)

In [ ]:
print(f"✅ Model Trained")
print(f"📈 RMSE: {rmse:.2f}")
print(f"📊 R² Score: {r2:.2f}")

In [ ]:
# import matplotlib.pyplot as plt

print("Train R²:", model.score(X_train, y_train))
print("Val R²:", model.score(X_val, y_val))
print("Test R²:", model.score(X_test, y_test))
